# 🎧 MagL+X: PyTorch Loss Sonification (Runnable Notebook)

This notebook demonstrates how to:
- Train a simple PyTorch model
- Extract per-sample losses
- Convert them into audio
- Stream real-time sound using MagL+X concepts

---


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch numpy sounddevice

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import sounddevice as sd

## 🔊 Synth Engine

In [ ]:
from synth import Synth

synth = Synth()

## 🎛 Live Audio Engine

In [ ]:
from live_engine import LiveEngine, CrossEntropyTracker

engine = LiveEngine()

## 🧠 Loss → Audio Mapping

In [ ]:
tracker = CrossEntropyTracker()

## 🧪 Toy Dataset

In [ ]:
# Simple synthetic classification data
X = torch.randn(200, 10)
y = (X.sum(dim=1) > 0).long()

## 🤖 Model

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(reduction='none')

## ▶️ Training with Live Audio

In [ ]:
engine.start()

for epoch in range(5):
    logits = model(X)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.mean().backward()
    optimizer.step()

    # choose either:
    # audio_fn = loss_to_audio_fn(loss)
    audio_fn = tracker.to_audio_fn(loss)

    engine.current_audio_fn = audio_fn

    print(f"Epoch {epoch}, loss = {loss.mean().item():.4f}")

engine.stop()